# 📊 Análisis Exploratorio - Gestión Inmobiliaria

Este notebook analiza las hojas del archivo Excel:
- **Operaciones**
- **Comisiones-equipo**
- **Cuotas-comisiones**
- **Clientes**

## Objetivo
Procesar, limpiar, depurar, transformar y analizar los datos para generar informes ejecutivos en formato tabular.


## 1. Importación de librerías


In [202]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Librerías importadas correctamente")


✅ Librerías importadas correctamente


## 2. Carga de las hojas del Excel


In [203]:
# Ruta al archivo Excel
excel_path = '../data/gestion-inmobiliaria.xlsx'

# Definir las hojas que vamos a analizar (con los nombres exactos del Excel)
hojas_objetivo = ['Operaciones', 'Comisiones-equipo', 'Cuotas-comisiones', 'Clientes']

# Diccionario para almacenar los DataFrames de cada hoja
dataframes = {}
errores_carga = {}

print(f"📁 Archivo: {excel_path}")
print(f"📋 Hojas objetivo: {', '.join(hojas_objetivo)}\n")

# Cargar todas las hojas
for hoja in hojas_objetivo:
    try:
        df_temp = pd.read_excel(excel_path, sheet_name=hoja)
        
        # Para Comisiones-equipo: detectar y quedarse solo con la primera tabla
        if hoja == 'Comisiones-equipo':
            # Buscar la primera fila completamente vacía que indica el fin de la primera tabla
            # o buscar patrones que indiquen el inicio de una segunda tabla
            filas_vacias = df_temp.isnull().all(axis=1)
            indices_vacias = df_temp[filas_vacias].index.tolist()
            
            # Si encontramos filas vacías, tomar solo hasta la primera fila vacía
            if indices_vacias:
                primera_fila_vacia = indices_vacias[0]
                df_temp = df_temp.iloc[:primera_fila_vacia]
                print(f"  ℹ️  Detectadas múltiples tablas. Se mantiene solo la primera tabla (hasta fila {primera_fila_vacia})")
            
            # También verificar si hay columnas que indiquen múltiples tablas (columnas completamente vacías en medio)
            # Eliminar columnas completamente vacías que puedan ser separadores
            columnas_vacias = df_temp.isnull().all(axis=0)
            if columnas_vacias.any():
                df_temp = df_temp.loc[:, ~columnas_vacias]
                print(f"  ℹ️  Eliminadas columnas vacías que separaban tablas")
        
        dataframes[hoja] = df_temp
        print(f"✅ {hoja}: {df_temp.shape[0]} filas × {df_temp.shape[1]} columnas")
    except Exception as e:
        errores_carga[hoja] = str(e)
        print(f"❌ Error al cargar '{hoja}': {e}")

print(f"\n📊 Total de hojas cargadas: {len(dataframes)}/{len(hojas_objetivo)}")

# Verificar hojas disponibles en el Excel
if len(errores_carga) > 0:
    excel_file = pd.ExcelFile(excel_path)
    print(f"\n📑 Hojas disponibles en el Excel: {excel_file.sheet_names}")


📁 Archivo: ../data/gestion-inmobiliaria.xlsx
📋 Hojas objetivo: Operaciones, Comisiones-equipo, Cuotas-comisiones, Clientes

✅ Operaciones: 14 filas × 14 columnas
  ℹ️  Eliminadas columnas vacías que separaban tablas
✅ Comisiones-equipo: 31 filas × 12 columnas
✅ Cuotas-comisiones: 18 filas × 5 columnas
✅ Clientes: 13 filas × 6 columnas

📊 Total de hojas cargadas: 4/4


## 3. Limpieza de Datos


In [204]:
# Definir columnas a eliminar por hoja
columnas_a_eliminar = {
    'Operaciones': ['Observaciones'],
    'Comisiones-equipo': ['Rol', 'Agentes', '$', 'Rol.1', 'Tipo operación', '% comisión'],
    'Cuotas-comisiones': ['Metodo de pago'],
    'Clientes': ['Teléfono', 'DNI o CUIT']
}

print("🔧 INICIANDO LIMPIEZA DE DATOS")
print("=" * 80)

# Procesar cada hoja
for nombre_hoja, df in dataframes.items():
    filas_antes = len(df)
    columnas_antes = len(df.columns)
    
    # 1. Eliminar filas completamente vacías
    df = df.dropna(how='all')
    
    # 2. Eliminar filas sin Nº Operación (verificar ambos nombres posibles de la columna)
    # Algunas hojas usan "Nº Operación" y otras "Nº de operación"
    columna_operacion = None
    if 'Nº Operación' in df.columns:
        columna_operacion = 'Nº Operación'
    elif 'Nº de operación' in df.columns:
        columna_operacion = 'Nº de operación'
    
    if columna_operacion:
        filas_antes_operacion = len(df)
        # Eliminar filas donde la columna de operación sea NaN o esté vacía
        df = df[df[columna_operacion].notna()].copy()
        # También eliminar si el valor es 0 o está vacío como string
        if df[columna_operacion].dtype == 'object':
            df = df[df[columna_operacion].astype(str).str.strip() != ''].copy()
        filas_eliminadas_operacion = filas_antes_operacion - len(df)
        if filas_eliminadas_operacion > 0:
            print(f"✓ {nombre_hoja}: Eliminadas {filas_eliminadas_operacion} filas sin {columna_operacion}")
    
    filas_eliminadas = filas_antes - len(df)
    
    # 2. Eliminar columnas específicas por hoja
    if nombre_hoja in columnas_a_eliminar:
        columnas_eliminar = columnas_a_eliminar[nombre_hoja]
        columnas_encontradas = [col for col in columnas_eliminar if col in df.columns]
        if columnas_encontradas:
            df = df.drop(columns=columnas_encontradas)
            print(f"✓ {nombre_hoja}: Eliminadas {len(columnas_encontradas)} columnas: {', '.join(columnas_encontradas)}")
        else:
            print(f"⚠ {nombre_hoja}: No se encontraron las columnas a eliminar")
    
    if filas_eliminadas > 0:
        print(f"✓ {nombre_hoja}: Eliminadas {filas_eliminadas} filas vacías")
    
    # Actualizar el DataFrame en el diccionario
    dataframes[nombre_hoja] = df
    
    columnas_despues = len(df.columns)
    print(f"  → Dimensiones finales: {len(df)} filas × {columnas_despues} columnas\n")

print("✅ Limpieza completada para todas las hojas")


🔧 INICIANDO LIMPIEZA DE DATOS
✓ Operaciones: Eliminadas 1 filas sin Nº Operación
✓ Operaciones: Eliminadas 1 columnas: Observaciones
✓ Operaciones: Eliminadas 1 filas vacías
  → Dimensiones finales: 13 filas × 13 columnas

✓ Comisiones-equipo: Eliminadas 1 filas sin Nº Operación
✓ Comisiones-equipo: Eliminadas 6 columnas: Rol, Agentes, $, Rol.1, Tipo operación, % comisión
✓ Comisiones-equipo: Eliminadas 1 filas vacías
  → Dimensiones finales: 30 filas × 6 columnas

✓ Cuotas-comisiones: Eliminadas 1 columnas: Metodo de pago
  → Dimensiones finales: 18 filas × 4 columnas

✓ Clientes: Eliminadas 2 columnas: Teléfono, DNI o CUIT
  → Dimensiones finales: 13 filas × 4 columnas

✅ Limpieza completada para todas las hojas


## 4. Resumen de Transformaciones Aplicadas


In [205]:
# Crear resumen de transformaciones
resumen_transformaciones = []

for nombre_hoja, df in dataframes.items():
    # Contar filas vacías que se eliminaron (simulando el proceso anterior)
    df_temp = pd.read_excel(excel_path, sheet_name=nombre_hoja)
    filas_antes = len(df_temp)
    filas_despues = len(df)
    filas_eliminadas = filas_antes - filas_despues
    
    # Contar columnas eliminadas
    columnas_antes = len(df_temp.columns)
    columnas_despues = len(df.columns)
    columnas_eliminadas = columnas_antes - columnas_despues
    
    # Detalle de columnas eliminadas
    if nombre_hoja in columnas_a_eliminar:
        columnas_elim = columnas_a_eliminar[nombre_hoja]
        detalle = ', '.join(columnas_elim)
    else:
        detalle = 'Ninguna'
    
    resumen_transformaciones.append({
        'Hoja': nombre_hoja,
        'Filas Antes': filas_antes,
        'Filas Después': filas_despues,
        'Filas Eliminadas': filas_eliminadas,
        'Columnas Antes': columnas_antes,
        'Columnas Después': columnas_despues,
        'Columnas Eliminadas': columnas_eliminadas,
        'Columnas Eliminadas (Detalle)': detalle
    })

df_resumen = pd.DataFrame(resumen_transformaciones)
print("📊 RESUMEN DE TRANSFORMACIONES")
print("=" * 80)
display(df_resumen)


📊 RESUMEN DE TRANSFORMACIONES


,Hoja,Filas Antes,Filas Después,Filas Eliminadas,Columnas Antes,Columnas Después,Columnas Eliminadas,Columnas Eliminadas (Detalle)
0,Operaciones,14,13,1,14,13,1,Observaciones
1,Comisiones-equipo,31,30,1,14,6,8,"Rol, Agentes, $, Rol.1, Tipo operación, % comi..."
2,Cuotas-comisiones,18,18,0,5,4,1,Metodo de pago
3,Clientes,13,13,0,6,4,2,"Teléfono, DNI o CUIT"


## 5. Normalización de Datos


In [206]:
# Definir columnas de montos por hoja (para convertir a formato de pesos argentinos)
columnas_montos = {
    'Operaciones': ['Comisión total', 'Cobrado', 'Saldo'],
    'Comisiones-equipo': ['Monto comisión', 'Saldo a pagar'],
    'Cuotas-comisiones': ['Importe'],
    'Clientes': []  # No tiene columnas de monto
}

# Definir columnas numéricas pequeñas que deben ser enteros
columnas_enteros = {
    'Operaciones': ['Nº Operación', 'Cuotas', 'Días estimados'],
    'Comisiones-equipo': ['Nº Operación'],
    'Cuotas-comisiones': ['Nº Operación', 'Cuota'],
    'Clientes': ['Edad', 'Nº de operación']
}

# Asegurar que Días estimados sea entero (ya está en la lista, pero lo confirmamos)

print("🔧 INICIANDO NORMALIZACIÓN DE DATOS")
print("=" * 80)

# Función para limpiar y convertir montos
def limpiar_monto(valor):
    """Convierte valores de monto a float, manejando strings con símbolos y espacios"""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    if isinstance(valor, str):
        # Remover símbolos de moneda, espacios y otros caracteres
        valor_limpio = valor.replace('$', '').replace(',', '').replace('.', '').replace(' ', '').strip()
        try:
            return float(valor_limpio)
        except:
            return np.nan
    return np.nan

# Procesar cada hoja
for nombre_hoja, df in dataframes.items():
    print(f"\n📋 Procesando: {nombre_hoja}")
    
    # 1. Normalizar columnas de montos
    if nombre_hoja in columnas_montos:
        for col_monto in columnas_montos[nombre_hoja]:
            if col_monto in df.columns:
                # Convertir a numérico
                df[col_monto] = df[col_monto].apply(limpiar_monto)
                # Redondear a 2 decimales
                df[col_monto] = df[col_monto].round(2)
                print(f"  ✓ {col_monto}: Convertido a formato numérico (2 decimales)")
    
    # 2. Normalizar columnas numéricas pequeñas a enteros
    if nombre_hoja in columnas_enteros:
        for col_entero in columnas_enteros[nombre_hoja]:
            if col_entero in df.columns:
                # Convertir a entero, manejando NaN
                df[col_entero] = pd.to_numeric(df[col_entero], errors='coerce').astype('Int64')
                print(f"  ✓ {col_entero}: Convertido a entero")
    
    # Actualizar el DataFrame
    dataframes[nombre_hoja] = df

print("\n✅ Normalización completada para todas las hojas")


🔧 INICIANDO NORMALIZACIÓN DE DATOS

📋 Procesando: Operaciones
  ✓ Comisión total: Convertido a formato numérico (2 decimales)
  ✓ Cobrado: Convertido a formato numérico (2 decimales)
  ✓ Saldo: Convertido a formato numérico (2 decimales)
  ✓ Nº Operación: Convertido a entero
  ✓ Cuotas: Convertido a entero
  ✓ Días estimados: Convertido a entero

📋 Procesando: Comisiones-equipo
  ✓ Monto comisión: Convertido a formato numérico (2 decimales)
  ✓ Saldo a pagar: Convertido a formato numérico (2 decimales)
  ✓ Nº Operación: Convertido a entero

📋 Procesando: Cuotas-comisiones
  ✓ Importe: Convertido a formato numérico (2 decimales)
  ✓ Nº Operación: Convertido a entero
  ✓ Cuota: Convertido a entero

📋 Procesando: Clientes
  ✓ Edad: Convertido a entero
  ✓ Nº de operación: Convertido a entero

✅ Normalización completada para todas las hojas


## 6. Vista Previa de Datos Normalizados


In [207]:
# Mostrar vista previa de cada hoja normalizada
for nombre_hoja, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📋 HOJA: {nombre_hoja.upper()}")
    print(f"{'='*80}")
    print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"\nTipos de datos:")
    for col in df.columns:
        tipo = df[col].dtype
        print(f"  - {col}: {tipo}")
    print(f"\nPrimeras 5 filas:")
    display(df.head())



📋 HOJA: OPERACIONES
Dimensiones: 13 filas × 13 columnas

Tipos de datos:
  - Nº Operación: Int64
  - Tipo: object
  - Codigo propiedad: object
  - Nombre propiedad: object
  - Cliente: object
  - Fecha inicio: datetime64[ns]
  - Fecha cierre: datetime64[ns]
  - Agente: object
  - Comisión total: float64
  - Cuotas: Int64
  - Saldo: float64
  - Cobrado: float64
  - Días estimados: Int64

Primeras 5 filas:


,Nº Operación,Tipo,Codigo propiedad,Nombre propiedad,Cliente,Fecha inicio,Fecha cierre,Agente,Comisión total,Cuotas,Saldo,Cobrado,Días estimados
0,1,alquiler,GARC1-154,Bonaire,Agustin Ruiz,2025-07-17,2025-07-28,Federico Trim,666666.66,1,0.0,666666.66,11
1,2,alquiler,GARC1-166,JR vidal 1768,maria antonela garrido,2025-08-02,2025-08-12,Alexis,430000.00,2,0.0,430000.00,10
2,3,alquiler,GARC1-93,25 de mayo aureliano,agustin martinez,2025-08-05,2025-08-08,Federico Trim,500000.00,2,0.0,500000.00,3
3,4,alquiler,GARC1-116,departamento calle uruguay,jacobo maria elena,2025-08-30,2025-09-09,Alexis,350000.00,2,0.0,350000.00,10
4,5,alquiler,GARC1-182,departamento calle alberdi,aloy marcos adrian,2025-09-03,2025-09-09,Federico Trim,480000.00,2,0.0,480000.00,6



📋 HOJA: COMISIONES-EQUIPO
Dimensiones: 30 filas × 6 columnas

Tipos de datos:
  - Nº Operación: Int64
  - Agente: object
  - Monto comisión: float64
  - Pagado: object
  - Porcentaje pagado: float64
  - Saldo a pagar: float64

Primeras 5 filas:


,Nº Operación,Agente,Monto comisión,Pagado,Porcentaje pagado,Saldo a pagar
0,1,Federico Trim,150000.0,Si,1.0,0.0
1,1,Juan Ignacio,150000.0,Si,1.0,0.0
2,1,Joaquin Garcia,50000.0,Si,1.0,0.0
3,2,Alexis,96750.0,Si,1.0,0.0
4,2,Juan Ignacio,96750.0,Si,1.0,0.0



📋 HOJA: CUOTAS-COMISIONES
Dimensiones: 18 filas × 4 columnas

Tipos de datos:
  - Nº Operación: Int64
  - Fecha: datetime64[ns]
  - Cuota: Int64
  - Importe: float64

Primeras 5 filas:


,Nº Operación,Fecha,Cuota,Importe
0,1,2025-07-17,1,666666.66
1,2,2025-08-12,1,215000.00
2,2,2025-09-10,2,215000.00
3,3,2025-08-08,1,250000.00
4,3,2025-09-10,2,250000.00



📋 HOJA: CLIENTES
Dimensiones: 13 filas × 4 columnas

Tipos de datos:
  - Nombre: object
  - Fecha nacimiento: datetime64[ns]
  - Edad: Int64
  - Nº de operación: Int64

Primeras 5 filas:


,Nombre,Fecha nacimiento,Edad,Nº de operación
0,Agustin Ruiz,2005-07-03,20,1
1,maria antonela garrido,1996-12-12,28,2
2,agustin martinez,1999-11-03,26,3
3,jacobo maria elena,1970-11-22,55,4
4,aloy marcos adrian,1981-01-13,44,5


## 7. Análisis: Hoja Operaciones


In [208]:
# Análisis de la hoja Operaciones
if 'Operaciones' in dataframes:
    df_ops = dataframes['Operaciones']
    
    print("📊 REPORTE: HOJA OPERACIONES")
    print("=" * 80)
    
    # 1. Reporte de lo cobrado hasta ahora
    total_cobrado = df_ops['Cobrado'].sum() if 'Cobrado' in df_ops.columns else 0
    total_comision = df_ops['Comisión total'].sum() if 'Comisión total' in df_ops.columns else 0
    total_saldo = df_ops['Saldo'].sum() if 'Saldo' in df_ops.columns else 0
    
    print(f"\n💰 RESUMEN FINANCIERO")
    print(f"  Total Comisión: ${total_comision:,.2f}")
    print(f"  Total Cobrado: ${total_cobrado:,.2f}")
    print(f"  Total Saldo Pendiente: ${total_saldo:,.2f}")
    print(f"  Porcentaje Cobrado: {(total_cobrado/total_comision*100):.2f}%" if total_comision > 0 else "  Porcentaje Cobrado: 0%")
    
    # 2. Indicadores: Cantidad de Alquileres y Ventas
    if 'Tipo' in df_ops.columns:
        tipo_counts = df_ops['Tipo'].value_counts()
        cantidad_alquileres = tipo_counts.get('alquiler', 0)
        cantidad_ventas = tipo_counts.get('venta', 0)
        
        print(f"\n📈 INDICADORES POR TIPO DE OPERACIÓN")
        print(f"  Alquileres: {cantidad_alquileres}")
        print(f"  Ventas: {cantidad_ventas}")
        print(f"  Total Operaciones: {len(df_ops)}")
    
    # 3. Tabla con operaciones que tienen saldo a cobrar
    if 'Saldo' in df_ops.columns:
        operaciones_pendientes = df_ops[df_ops['Saldo'] > 0].copy()
        
        if len(operaciones_pendientes) > 0:
            print(f"\n📋 OPERACIONES CON SALDO A COBRAR ({len(operaciones_pendientes)} operaciones)")
            print("=" * 80)
            
            # Seleccionar columnas relevantes para mostrar
            columnas_mostrar = ['Nº Operación', 'Tipo', 'Cliente', 'Comisión total', 'Cobrado', 'Saldo']
            columnas_disponibles = [col for col in columnas_mostrar if col in operaciones_pendientes.columns]
            
            # Formatear montos para mostrar
            df_mostrar = operaciones_pendientes[columnas_disponibles].copy()
            for col in ['Comisión total', 'Cobrado', 'Saldo']:
                if col in df_mostrar.columns:
                    df_mostrar[col] = df_mostrar[col].apply(lambda x: f"${x:,.2f}" if pd.notna(x) else "$0.00")
            
            display(df_mostrar)
        else:
            print(f"\n✅ No hay operaciones con saldo pendiente")
else:
    print("⚠️ La hoja 'Operaciones' no está disponible")


📊 REPORTE: HOJA OPERACIONES

💰 RESUMEN FINANCIERO
  Total Comisión: $8,376,666.66
  Total Cobrado: $6,426,666.66
  Total Saldo Pendiente: $1,950,000.00
  Porcentaje Cobrado: 76.72%

📈 INDICADORES POR TIPO DE OPERACIÓN
  Alquileres: 13
  Ventas: 0
  Total Operaciones: 13

📋 OPERACIONES CON SALDO A COBRAR (4 operaciones)


,Nº Operación,Tipo,Cliente,Comisión total,Cobrado,Saldo
9,10,alquiler,alarcon mauro agustin,"$400,000.00","$200,000.00","$200,000.00"
10,11,alquiler,alfredo oscar silva,"$1,000,000.00","$500,000.00","$500,000.00"
11,12,alquiler,hermida rocio ivon,"$650,000.00",$0.00,"$650,000.00"
12,13,alquiler,Corven sa,"$600,000.00",$0.00,"$600,000.00"


## 8. Análisis: Hoja Comisiones-equipo


In [209]:
# Análisis de la hoja Comisiones-equipo
if 'Comisiones-equipo' in dataframes:
    df_com = dataframes['Comisiones-equipo']
    
    print("📊 REPORTE: HOJA COMISIONES-EQUIPO")
    print("=" * 80)
    
    # Verificar que tenemos las columnas necesarias
    columnas_necesarias = ['Agente', 'Monto comisión', 'Pagado', 'Porcentaje pagado']
    columnas_faltantes = [col for col in columnas_necesarias if col not in df_com.columns]
    
    if columnas_faltantes:
        print(f"⚠️ Faltan las siguientes columnas: {', '.join(columnas_faltantes)}")
    else:
        # Calcular lo pendiente por cobrar para cada registro
        # Pendiente = Monto comisión si Pagado = "NO" 
        # O si Pagado = "SI" y Porcentaje pagado < 1.0, entonces pendiente = Monto comisión * (1 - Porcentaje pagado)
        
        def calcular_pendiente(row):
            monto = row['Monto comisión']
            pagado = str(row['Pagado']).strip().upper() if pd.notna(row['Pagado']) else ''
            porcentaje = row['Porcentaje pagado'] if pd.notna(row['Porcentaje pagado']) else 0
            
            if pagado == 'NO':
                return monto
            elif pagado == 'SI' and porcentaje < 1.0:
                return monto * (1 - porcentaje)
            else:
                return 0.0
        
        df_com['Pendiente'] = df_com.apply(calcular_pendiente, axis=1)
        
        # Calcular lo pagado para cada registro
        def calcular_pagado(row):
            monto = row['Monto comisión']
            pagado = str(row['Pagado']).strip().upper() if pd.notna(row['Pagado']) else ''
            porcentaje = row['Porcentaje pagado'] if pd.notna(row['Porcentaje pagado']) else 0
            
            if pagado == 'SI':
                return monto * porcentaje
            else:
                return 0.0
        
        df_com['Pagado_calculado'] = df_com.apply(calcular_pagado, axis=1)
        
        # Filtrar registros: eliminar los que no tienen Nº Operación o agente vacío
        # Primero eliminar filas sin Nº Operación (ya debería estar hecho en limpieza, pero por si acaso)
        columna_operacion = None
        if 'Nº Operación' in df_com.columns:
            columna_operacion = 'Nº Operación'
        elif 'Nº de operación' in df_com.columns:
            columna_operacion = 'Nº de operación'
        
        if columna_operacion:
            df_com_limpio = df_com[df_com[columna_operacion].notna()].copy()
            if df_com_limpio[columna_operacion].dtype == 'object':
                df_com_limpio = df_com_limpio[df_com_limpio[columna_operacion].astype(str).str.strip() != ''].copy()
        else:
            df_com_limpio = df_com.copy()
        
        # Luego eliminar registros con agente vacío o NaN
        df_com_limpio = df_com_limpio[df_com_limpio['Agente'].notna() & (df_com_limpio['Agente'].astype(str).str.strip() != '')].copy()
        
        if len(df_com_limpio) < len(df_com):
            registros_filtrados = len(df_com) - len(df_com_limpio)
            print(f"  ⚠️  Se filtraron {registros_filtrados} registros sin {columna_operacion if columna_operacion else 'Nº Operación'} o con agente vacío/inválido")
        
        # Agrupar por agente
        resumen_agentes = df_com_limpio.groupby('Agente').agg({
            'Monto comisión': 'sum',
            'Pagado_calculado': 'sum',
            'Pendiente': 'sum'
        }).reset_index()
        
        resumen_agentes.columns = ['Agente', 'Total Comisión', 'Total Pagado', 'Total Pendiente']
        
        # Redondear a 2 decimales
        resumen_agentes['Total Comisión'] = resumen_agentes['Total Comisión'].round(2)
        resumen_agentes['Total Pagado'] = resumen_agentes['Total Pagado'].round(2)
        resumen_agentes['Total Pendiente'] = resumen_agentes['Total Pendiente'].round(2)
        
        print(f"\n💰 RESUMEN POR AGENTE")
        print("=" * 80)
        display(resumen_agentes)
        
        # Resumen general
        total_comision = resumen_agentes['Total Comisión'].sum()
        total_pagado = resumen_agentes['Total Pagado'].sum()
        total_pendiente = resumen_agentes['Total Pendiente'].sum()
        
        print(f"\n📊 RESUMEN GENERAL")
        print(f"  Total Comisiones: ${total_comision:,.2f}")
        print(f"  Total Pagado: ${total_pagado:,.2f}")
        print(f"  Total Pendiente: ${total_pendiente:,.2f}")
        print(f"  Porcentaje Pagado: {(total_pagado/total_comision*100):.2f}%" if total_comision > 0 else "  Porcentaje Pagado: 0%")
else:
    print("⚠️ La hoja 'Comisiones-equipo' no está disponible")


📊 REPORTE: HOJA COMISIONES-EQUIPO

💰 RESUMEN POR AGENTE


,Agente,Total Comisión,Total Pagado,Total Pendiente
0,Alexis,659250.0,468000.0,191250.0
1,Federico Trim,1333500.0,973500.0,360000.0
2,Joaquin Garcia,50000.0,50000.0,0.0
3,Juan Ignacio,927750.0,594000.0,333750.0
4,Julian Meza,247500.0,247500.0,0.0



📊 RESUMEN GENERAL
  Total Comisiones: $3,218,000.00
  Total Pagado: $2,333,000.00
  Total Pendiente: $885,000.00
  Porcentaje Pagado: 72.50%


## 9. Análisis: Hoja Clientes


In [210]:
# Análisis de la hoja Clientes
if 'Clientes' in dataframes:
    df_clientes = dataframes['Clientes']
    
    print("📊 REPORTE: HOJA CLIENTES")
    print("=" * 80)
    
    # 1. Cantidad total de clientes (todos los que tienen Nº de operación)
    total_clientes = len(df_clientes)
    print(f"\n👥 CANTIDAD DE CLIENTES")
    print(f"  Total de clientes: {total_clientes}")
    
    # 2. Análisis de edad (ignorando edades > 70 para cálculos, pero contando el cliente)
    if 'Edad' in df_clientes.columns:
        # Filtrar edades válidas (<= 70) para el promedio
        edades_validas = df_clientes[df_clientes['Edad'] <= 70]['Edad']
        edades_invalidas = df_clientes[df_clientes['Edad'] > 70]
        
        if len(edades_invalidas) > 0:
            print(f"  ⚠️  {len(edades_invalidas)} clientes con edad > 70 (ignorados en promedio, pero contados en total)")
        
        if len(edades_validas) > 0:
            promedio_edad = edades_validas.mean()
            print(f"  Promedio de edad (excluyendo > 70): {promedio_edad:.1f} años")
            print(f"  Edad mínima: {edades_validas.min()} años")
            print(f"  Edad máxima: {edades_validas.max()} años")
        else:
            print(f"  ⚠️  No hay edades válidas para calcular el promedio")
        
        # 3. Rango etario de clientes
        print(f"\n📊 RANGO ETARIO DE CLIENTES")
        
        # Definir rangos etarios
        def clasificar_edad(edad):
            if pd.isna(edad) or edad > 70:
                return 'Sin clasificar (>70 o sin dato)'
            elif edad < 25:
                return 'Menos de 25 años'
            elif edad < 35:
                return '25-34 años'
            elif edad < 45:
                return '35-44 años'
            elif edad < 55:
                return '45-54 años'
            elif edad <= 70:
                return '55-70 años'
            else:
                return 'Sin clasificar (>70 o sin dato)'
        
        df_clientes['Rango Etario'] = df_clientes['Edad'].apply(clasificar_edad)
        distribucion_etaria = df_clientes['Rango Etario'].value_counts().sort_index()
        
        # Crear tabla con distribución etaria
        tabla_etaria = pd.DataFrame({
            'Rango Etario': distribucion_etaria.index,
            'Cantidad': distribucion_etaria.values,
            'Porcentaje': (distribucion_etaria.values / total_clientes * 100).round(2)
        })
        
        display(tabla_etaria)
    else:
        print("  ⚠️  No se encontró la columna 'Edad'")
else:
    print("⚠️ La hoja 'Clientes' no está disponible")


📊 REPORTE: HOJA CLIENTES

👥 CANTIDAD DE CLIENTES
  Total de clientes: 13
  ⚠️  1 clientes con edad > 70 (ignorados en promedio, pero contados en total)
  Promedio de edad (excluyendo > 70): 35.4 años
  Edad mínima: 20 años
  Edad máxima: 59 años

📊 RANGO ETARIO DE CLIENTES


,Rango Etario,Cantidad,Porcentaje
0,25-34 años,6,46.15
1,35-44 años,1,7.69
2,45-54 años,1,7.69
3,55-70 años,2,15.38
4,Menos de 25 años,2,15.38
5,Sin clasificar (>70 o sin dato),1,7.69


## 10. Flujo de Caja - Informe Ejecutivo


In [211]:
# Flujo de Caja - Informe Ejecutivo
print("💰 FLUJO DE CAJA - INFORME EJECUTIVO")
print("=" * 80)

# 1. Lo cobrado (de Operaciones)
cobrado_total = 0
if 'Operaciones' in dataframes:
    df_ops = dataframes['Operaciones']
    if 'Cobrado' in df_ops.columns:
        cobrado_total = df_ops['Cobrado'].sum()

# 2. Lo por cobrar (de Operaciones - Saldo pendiente)
por_cobrar_total = 0
if 'Operaciones' in dataframes:
    df_ops = dataframes['Operaciones']
    if 'Saldo' in df_ops.columns:
        por_cobrar_total = df_ops['Saldo'].sum()

# 3. Lo por pagar (de Comisiones-equipo - Pendiente)
por_pagar_total = 0
if 'Comisiones-equipo' in dataframes:
    df_com = dataframes['Comisiones-equipo']
    # Calcular pendiente si no existe la columna
    if 'Pendiente' not in df_com.columns:
        def calcular_pendiente(row):
            monto = row['Monto comisión'] if 'Monto comisión' in row else 0
            pagado = str(row['Pagado']).strip().upper() if pd.notna(row.get('Pagado')) else ''
            porcentaje = row['Porcentaje pagado'] if pd.notna(row.get('Porcentaje pagado')) else 0
            
            if pagado == 'NO':
                return monto
            elif pagado == 'SI' and porcentaje < 1.0:
                return monto * (1 - porcentaje)
            else:
                return 0.0
        
        df_com['Pendiente'] = df_com.apply(calcular_pendiente, axis=1)
    
    # Filtrar registros válidos
    if 'Nº Operación' in df_com.columns:
        df_com_limpio = df_com[df_com['Nº Operación'].notna()].copy()
    else:
        df_com_limpio = df_com.copy()
    
    if 'Pendiente' in df_com_limpio.columns:
        por_pagar_total = df_com_limpio['Pendiente'].sum()

# 4. Lo pagado (de Comisiones-equipo - Ya pagado)
pagado_total = 0
if 'Comisiones-equipo' in dataframes:
    df_com = dataframes['Comisiones-equipo']
    # Calcular pagado si no existe la columna
    if 'Pagado_calculado' not in df_com.columns:
        def calcular_pagado(row):
            monto = row['Monto comisión'] if 'Monto comisión' in row else 0
            pagado = str(row['Pagado']).strip().upper() if pd.notna(row.get('Pagado')) else ''
            porcentaje = row['Porcentaje pagado'] if pd.notna(row.get('Porcentaje pagado')) else 0
            
            if pagado == 'SI':
                return monto * porcentaje
            else:
                return 0.0
        
        df_com['Pagado_calculado'] = df_com.apply(calcular_pagado, axis=1)
    
    # Filtrar registros válidos
    if 'Nº Operación' in df_com.columns:
        df_com_limpio = df_com[df_com['Nº Operación'].notna()].copy()
    else:
        df_com_limpio = df_com.copy()
    
    if 'Pagado_calculado' in df_com_limpio.columns:
        pagado_total = df_com_limpio['Pagado_calculado'].sum()

# 5. Balance neto = Lo cobrado - Lo pagado
balance_neto = cobrado_total - pagado_total

# 6. Flujo futuro neto = Lo por cobrar - Lo por pagar
flujo_futuro_neto = por_cobrar_total - por_pagar_total

# Crear tabla resumen
resumen_flujo_caja = pd.DataFrame({
    'Concepto': [
        'Total Cobrado (Operaciones)',
        'Total Pagado (Comisiones)',
        'Por Cobrar (Saldo Operaciones)',
        'Por Pagar (Comisiones Pendientes)',
        'Balance Neto (Cobrado - Pagado)',
        'Flujo Futuro Neto (Por Cobrar - Por Pagar)'
    ],
    'Monto': [
        cobrado_total,
        pagado_total,
        por_cobrar_total,
        por_pagar_total,
        balance_neto,
        flujo_futuro_neto
    ]
})

# Formatear montos
resumen_flujo_caja['Monto Formateado'] = resumen_flujo_caja['Monto'].apply(lambda x: f"${x:,.2f}")

print("\n📊 RESUMEN FINANCIERO")
print("=" * 80)
display(resumen_flujo_caja[['Concepto', 'Monto Formateado']].rename(columns={'Monto Formateado': 'Monto'}))

print(f"\n💡 INTERPRETACIÓN:")
print(f"  • Lo que ya se cobró de operaciones: ${cobrado_total:,.2f}")
print(f"  • Lo que ya se pagó en comisiones: ${pagado_total:,.2f}")
print(f"  • Lo que falta cobrar de operaciones: ${por_cobrar_total:,.2f}")
print(f"  • Lo que falta pagar en comisiones: ${por_pagar_total:,.2f}")
print(f"  • Balance neto actual: ${balance_neto:,.2f}")
print(f"  • Flujo futuro neto esperado: ${flujo_futuro_neto:,.2f}")


💰 FLUJO DE CAJA - INFORME EJECUTIVO

📊 RESUMEN FINANCIERO


,Concepto,Monto
0,Total Cobrado (Operaciones),"$6,426,666.66"
1,Total Pagado (Comisiones),"$2,333,000.00"
2,Por Cobrar (Saldo Operaciones),"$1,950,000.00"
3,Por Pagar (Comisiones Pendientes),"$885,000.00"
4,Balance Neto (Cobrado - Pagado),"$4,093,666.66"
5,Flujo Futuro Neto (Por Cobrar - Por Pagar),"$1,065,000.00"



💡 INTERPRETACIÓN:
  • Lo que ya se cobró de operaciones: $6,426,666.66
  • Lo que ya se pagó en comisiones: $2,333,000.00
  • Lo que falta cobrar de operaciones: $1,950,000.00
  • Lo que falta pagar en comisiones: $885,000.00
  • Balance neto actual: $4,093,666.66
  • Flujo futuro neto esperado: $1,065,000.00
